# Calculate Curve Length

Computes the total arc length of a cam profile curve from an ordered set of
(x, y) points stored in a CSV file. The points are fit with a parametric
spline and the spline's length is estimated by numerical integration,
yielding a single scalar (in the same units as the input coordinates,
typically meters).

This is a standalone utility for spot-checking the total unwrapped length of
a cam profile (e.g. cable travel over a full rotation), independent of the
per-point cumulative-length helper in [class_cam_generation.py](../class_cam_generation.py).

**Inputs:** a CSV file of ordered cam profile points (default:
`data/inner_63_stroke.csv`).
**Outputs:** a printed/inspectable scalar curve length; no files are written.

In [ ]:
import numpy as np
from scipy.interpolate import make_interp_spline
from scipy.integrate import trapezoid

from class_cam_generation import read_xy_from_csv


In [ ]:
def curve_length_from_csv(csv_filepath, n_eval=1000, spline_order=3,
                          x_col=0, y_col=1, delimiter=','):
    """Compute the total length of a spline curve through ordered CSV points.

    Unlike class_cam_generation.curve_length_from_csv, which returns the
    cumulative length at each input point, this returns a single scalar: the
    total length of the fitted curve.

    The function reads x/y points from the provided CSV file, constructs a
    parametric spline curve that passes through the points in order, and
    integrates the curve length.

    Parameters:
    csv_filepath: str
        Path to the CSV file containing ordered x and y coordinates.
    n_eval: int, optional
        Number of points to evaluate along the spline for length estimation.
    spline_order: int, optional
        Order of the spline. Default is 3 (cubic) and is automatically reduced
        when there are too few data points.
    x_col: int, optional
        Index of the column containing x coordinates.
    y_col: int, optional
        Index of the column containing y coordinates.
    delimiter: str, optional
        Field delimiter used by the CSV file.

    Returns:
    float
        Estimated total length of the interpolated curve.
    """
    x, y = read_xy_from_csv(csv_filepath, x_col=x_col, y_col=y_col,
                            delimiter=delimiter)
    if x.size < 2:
        return 0.0
    t = np.linspace(0.0, 1.0, x.size)
    k = min(spline_order, x.size - 1)
    spline = make_interp_spline(t, np.column_stack((x, y)), k=k, axis=0)
    t_eval = np.linspace(t[0], t[-1], n_eval)
    derivative = spline.derivative()
    dx_dy = derivative(t_eval)
    speeds = np.hypot(dx_dy[:, 0], dx_dy[:, 1])
    return float(trapezoid(speeds, t_eval))

In [6]:
csv_filepath = 'data/inner_63_stroke.csv'
result = curve_length_from_csv(csv_filepath, n_eval=1000, spline_order=3, x_col=0, y_col=1, delimiter=',')